In [ ]:
import http.server
import socketserver
import datetime
import pandas as pd
import json
import asyncio
import aiohttp



# Exercise 1


In [ ]:
 
class DataProvider:
    def __init__(self, data_file='dSST.csv'):
        self.data = pd.read_csv(data_file)

    def get_data(self, parameter=None):
        if parameter is None:
            return self.data.to_json(orient='records')

        if isinstance(parameter, int):
            year_data = self.data.loc[self.data['Year'] == parameter]
            if year_data.empty:
                raise ValueError("No data found for the specified year")
            return year_data.to_json(orient='records')

        if isinstance(parameter, list) and len(parameter) == 2:
            start_year, end_year = parameter
            year_data = self.data.loc[(self.data['Year'] >= start_year) & (self.data['Year'] <= end_year)]
            if year_data.empty:
                raise ValueError("No data found for the specified year range")
            return year_data.to_json(orient='records')

        raise ValueError("Invalid parameter")

d = DataProvider()
print(d.get_data()) 
print(d.get_data(1991)) 
print(d.get_data([1991,2000])) 

In [ ]:
 
PORT = 8000

class CustomRequestHandler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        self.data_provider = DataProvider()
        super().__init__(*args, **kwargs)
        
    def go_GET(self):
        if self.path == '/data':
            self.handle_data_request()
        else:
            self.send_error(404, "Not found")

    def handle_data_request(self):
        if self.path.endswith('/all') or self.is_valid_date_request():
            # Process the request and send the response
            self.send_response(200)
            self.send_header('Content-type', 'text/plain')
            self.end_headers() 
            self.wfile.write(b'Yeay')
        else:
            self.send_error(400, "Bad Request")

    def is_valid_date_request(self): 
        # Check if the request path contains one or two dates
        path_parts = self.path.split('/')
        if len(path_parts) == 3:  # Assumes the format /data/YYYY-MM-DD
            date_str = path_parts[2]
            try:
                datetime.datetime.strptime(date_str, '%Y-%m-%d')
                return True
            except ValueError:
                return False
        elif len(path_parts) == 4:  # Assumes the format /data/YYYY-MM-DD/YYYY-MM-DD
            date_str1 = path_parts[2]
            date_str2 = path_parts[3]
            try:
                datetime.datetime.strptime(date_str1, '%Y-%m-%d')
                datetime.datetime.strptime(date_str2, '%Y-%m-%d')
                return True
            except ValueError:
                return False
        else:
            return False


handler = CustomRequestHandler
http = socketserver.TCPServer(("", PORT), handler)

print("serving at port", PORT)
http.serve_forever()
 
